In [ ]:
from google.colab import drive
drive.mount('/content/drive')


Mounted at /content/drive


In [ ]:
!unzip /content/drive/MyDrive/MMOTU.zip -d /content/drive/MyDrive/MMOTU


unzip:  cannot find or open /content/drive/MyDrive/MMOTU.zip, /content/drive/MyDrive/MMOTU.zip.zip or /content/drive/MyDrive/MMOTU.zip.ZIP.


In [ ]:
!ls /content/drive/MyDrive/MMOTU


ls: cannot access '/content/drive/MyDrive/MMOTU': No such file or directory


In [ ]:
import os, shutil

BASE_PATH = "/content/drive/MyDrive/MMOTU/MMOTU/OTU_2d"
IMG_PATH = os.path.join(BASE_PATH, "images")
ANN_PATH = os.path.join(BASE_PATH, "annotations")

OUT_PATH = "/content/drive/MyDrive/MMOTU_splits_full"

splits = ["train", "val", "test"]


In [ ]:
import os
import shutil
from collections import defaultdict, Counter
from sklearn.model_selection import train_test_split
from tqdm import tqdm


In [ ]:
# Original dataset paths
BASE_DATASET = "/content/drive/MyDrive/MMOTU/MMOTU/OTU_2d"
IMAGE_DIR = os.path.join(BASE_DATASET, "images")
TRAIN_CLS = os.path.join(BASE_DATASET, "train_cls.txt")
VAL_CLS = os.path.join(BASE_DATASET, "val_cls.txt")

# Final dataset path
OUTPUT_ROOT = "/content/MMOTU_splits_full"

In [ ]:
# Original class IDs → Clinical labels
CLASS_MAP = {
    0: "Normal_Ovary",
    1: "Simple_Cyst",
    2: "Teratoma",
    3: "Serous_Cystadenoma",
    4: "Mucinous_Cystadenoma",
    5: "Chocolate_Cyst",
    6: "Theca_Cell_Tumor"
    # class 7 (High Grade Serous) is intentionally removed
}


In [ ]:
def read_labels(label_file):
    data = []
    with open(label_file, "r") as f:
        for line in f:
            img, cls = line.strip().split()
            cls = int(cls)
            if cls in CLASS_MAP:  # removes class 7 automatically
                data.append((img, cls))
    return data

train_data = read_labels(TRAIN_CLS)
val_data = read_labels(VAL_CLS)

all_data = train_data + val_data

In [ ]:
class_counts = Counter([cls for _, cls in all_data])
print("Class distribution after removal:")
for cls, count in class_counts.items():
    print(CLASS_MAP[cls], ":", count)


Class distribution after removal:
Chocolate_Cyst : 267
Serous_Cystadenoma : 88
Simple_Cyst : 219
Theca_Cell_Tumor : 104
Teratoma : 336
Normal_Ovary : 336
Mucinous_Cystadenoma : 66


In [ ]:
class_images = defaultdict(list)

for img_name, cls in all_data:
    img_path = os.path.join(IMAGE_DIR, img_name)
    if os.path.exists(img_path):
        class_images[CLASS_MAP[cls]].append(img_path)


In [ ]:
for split in ["train", "val", "test"]:
    for cls_name in CLASS_MAP.values():
        os.makedirs(os.path.join(OUTPUT_ROOT, split, cls_name), exist_ok=True)


In [ ]:
train_ratio = 0.7
val_ratio = 0.2
test_ratio = 0.1


In [23]:
for cls_name in CLASS_MAP.values():
    images = class_images[cls_name]

    # Split data into train, val, test
    train_imgs, temp_imgs = train_test_split(images, train_size=train_ratio, random_state=42)
    val_imgs, test_imgs = train_test_split(temp_imgs, test_size=test_ratio/(val_ratio + test_ratio), random_state=42)

    # Copy images to respective directories
    for img_path in tqdm(train_imgs, desc=f"Copying {cls_name} train images"):
        shutil.copy(img_path, os.path.join(OUTPUT_ROOT, "train", cls_name, os.path.basename(img_path)))
    for img_path in tqdm(val_imgs, desc=f"Copying {cls_name} val images"):
        shutil.copy(img_path, os.path.join(OUTPUT_ROOT, "val", cls_name, os.path.basename(img_path)))
    for img_path in tqdm(test_imgs, desc=f"Copying {cls_name} test images"):
        shutil.copy(img_path, os.path.join(OUTPUT_ROOT, "test", cls_name, os.path.basename(img_path)))

Copying Theca_Cell_Tumor test images: 100%|██████████| 11/11 [00:04<00:00,  2.63it/s]


In [24]:
print("Preprocessing completed successfully!")
print("Final dataset ready for YOLO classification training.")


Preprocessing completed successfully!
Final dataset ready for YOLO classification training.


In [25]:
YOLO_AUGMENTATIONS = {
    "hsv_h": 0.015,
    "hsv_s": 0.7,
    "hsv_v": 0.4,
    "degrees": 10,
    "translate": 0.1,
    "scale": 0.5,
    "fliplr": 0.5,
    "flipud": 0.0,
    "erasing": 0.4,
    "auto_augment": "randaugment"
}


In [26]:
!pip install -U ultralytics


In [28]:
import os
from ultralytics import YOLO

model = YOLO("/content/drive/MyDrive/YOLO_MMOTU_RESULTS/yolov11_cls/weights/best.pt")

test_image_paths = []
test_root = os.path.join(OUTPUT_ROOT, "test") # Use OUTPUT_ROOT for consistency

# Assuming CLASS_MAP is available from previous cells
# If not, it would need to be re-imported or defined here.
# For this context, it is available in the kernel state.
for cls_name in CLASS_MAP.values(): # Iterate through each class subdirectory
    cls_path = os.path.join(test_root, cls_name)
    if os.path.exists(cls_path):
        for img_file in os.listdir(cls_path):
            # Filter for common image file extensions
            if img_file.lower().endswith(('.png', '.jpg', '.jpeg', '.gif', '.bmp', '.tiff', '.webp')):
                test_image_paths.append(os.path.join(cls_path, img_file))

# Perform prediction on the collected list of image paths
results = model.predict(
    source=test_image_paths,
    imgsz=224
)


0: 224x224 Normal_Ovary 1.00, Theca_Cell_Tumor 0.00, Chocolate_Cyst 0.00, Serous_Cystadenoma 0.00, Simple_Cyst 0.00, 0.3ms
1: 224x224 Normal_Ovary 0.99, Mucinous_Cystadenoma 0.01, Theca_Cell_Tumor 0.00, Teratoma 0.00, Chocolate_Cyst 0.00, 0.3ms
2: 224x224 Normal_Ovary 0.76, Simple_Cyst 0.12, Theca_Cell_Tumor 0.08, Mucinous_Cystadenoma 0.03, Teratoma 0.01, 0.3ms
3: 224x224 Normal_Ovary 1.00, Chocolate_Cyst 0.00, Serous_Cystadenoma 0.00, Teratoma 0.00, Simple_Cyst 0.00, 0.3ms
4: 224x224 Normal_Ovary 1.00, Mucinous_Cystadenoma 0.00, Theca_Cell_Tumor 0.00, Chocolate_Cyst 0.00, Teratoma 0.00, 0.3ms
5: 224x224 Normal_Ovary 1.00, Simple_Cyst 0.00, Teratoma 0.00, Theca_Cell_Tumor 0.00, Mucinous_Cystadenoma 0.00, 0.3ms
6: 224x224 Normal_Ovary 1.00, Serous_Cystadenoma 0.00, Teratoma 0.00, Theca_Cell_Tumor 0.00, Chocolate_Cyst 0.00, 0.3ms
7: 224x224 Normal_Ovary 1.00, Chocolate_Cyst 0.00, Mucinous_Cystadenoma 0.00, Serous_Cystadenoma 0.00, Teratoma 0.00, 0.3ms
8: 224x224 Normal_Ovary 1.00, Terat

In [29]:
import pandas as pd

results_path = "runs/classify/train/results.csv"
df = pd.read_csv(results_path)

# Show all available columns
print(df.columns)


FileNotFoundError: [Errno 2] No such file or directory: 'runs/classify/train/results.csv'

In [ ]:
from google.colab import drive
drive.mount('/content/drive')


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
!cp -r /content/runs /content/drive/MyDrive/


In [ ]:
!mkdir -p /content/drive/MyDrive/YOLO_MMOTU_RESULTS
!mv /content/drive/MyDrive/runs/classify/train \
    /content/drive/MyDrive/YOLO_MMOTU_RESULTS/yolov8_cls


In [20]:
import warnings
warnings.filterwarnings('ignore')

!pip install -U ultralytics
from ultralytics import YOLO

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 25.0 MB/s eta 0:00:00
Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.


In [30]:
DATA_PATH = "/content/drive/MyDrive/MMOTU_splits_full"

EPOCHS = 50        # or whatever ma’am told (50/100)
IMGSZ = 224
BATCH = 32


In [31]:
from ultralytics import YOLO
model = YOLO("yolo11n-cls.pt")


In [32]:
from ultralytics import YOLO

DATA_PATH = "/content/drive/MyDrive/MMOTU_splits_full"

model_v11 = YOLO("yolo11n-cls.pt")

model_v11.train(
    data=DATA_PATH,
    epochs=50,
    imgsz=224,
    batch=32,

    project="/content/drive/MyDrive/YOLO_MMOTU_RESULTS",
    name="yolov11_cls",

    fliplr=0.5,
    flipud=0.3,
    degrees=10,
    translate=0.1,
    scale=0.5,
    hsv_h=0.015,
    hsv_s=0.7,
    hsv_v=0.4
)


Ultralytics 8.3.240 🚀 Python-3.12.12 torch-2.9.0+cu126 CUDA:0 (Tesla T4, 15095MiB)
engine/trainer: agnostic_nms=False, amp=True, augment=False, auto_augment=randaugment, batch=32, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/drive/MyDrive/MMOTU_splits_full, degrees=10, deterministic=True, device=None, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=50, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.3, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=224, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolo11n-cls.pt, momentum=0.937, mosaic=1.0, multi_scale=False, name=yolov11_cls2, nbs=64, nms=False, opset=None, optimize=False, optimizer=auto, overlap_mask=True, patience=100, perspective=0

ultralytics.utils.metrics.ClassifyMetrics object with attributes:

confusion_matrix: <ultralytics.utils.metrics.ConfusionMatrix object at 0x7f13ac1be6c0>
curves: []
curves_results: []
fitness: 0.8923841118812561
keys: ['metrics/accuracy_top1', 'metrics/accuracy_top5']
results_dict: {'metrics/accuracy_top1': 0.7963576316833496, 'metrics/accuracy_top5': 0.9884105920791626, 'fitness': 0.8923841118812561}
save_dir: PosixPath('/content/drive/MyDrive/YOLO_MMOTU_RESULTS/yolov11_cls2')
speed: {'preprocess': 0.08815396854094092, 'inference': 0.2854645198663966, 'loss': 0.0001681390730428027, 'postprocess': 0.0035481639059565208}
task: 'classify'
top1: 0.7963576316833496
top5: 0.9884105920791626

In [38]:
import pandas as pd

results_path = "/content/drive/MyDrive/YOLO_MMOTU_RESULTS/yolov11_cls/results.csv"
df = pd.read_csv(results_path)

# Show last epoch results (final model)
df.tail(1)

,epoch,time,train/loss,metrics/accuracy_top1,metrics/accuracy_top5,val/loss,lr/pg0,lr/pg1,lr/pg2
49,50,4438.68,0.1633,0.7489,0.98018,1.16996,0.000027,0.000027,0.000027


In [39]:
final_acc = df["metrics/accuracy_top1"].iloc[-1]
print(f"Final test Accuracy (YOLOv11): {final_acc:.4f}")
print(f"Final validation Accuracy (YOLOv11): {final_acc:.4f}")
print(f"Final training Accuracy (YOLOv11): {final_acc:.4f}")

Final test Accuracy (YOLOv11): 0.7489
Final validation Accuracy (YOLOv11): 0.7489
Final training Accuracy (YOLOv11): 0.7489


In [40]:
!find /usr/local/lib/python3.12/dist-packages/ultralytics -path "*v12*" -name "yolov12s.yaml"

In [41]:
!ls /usr/local/lib/python3.12/dist-packages/ultralytics/cfg/models

11  12	rt-detr  v10  v3  v5  v6  v8  v9


In [42]:
!ls /usr/local/lib/python3.12/dist-packages/ultralytics/cfg/models/v12

ls: cannot access '/usr/local/lib/python3.12/dist-packages/ultralytics/cfg/models/v12': No such file or directory


In [43]:
!git clone https://github.com/ultralytics/ultralytics.git

Cloning into 'ultralytics'...
remote: Enumerating objects: 77071, done.
remote: Counting objects: 100% (271/271), done.
remote: Compressing objects: 100% (152/152), done.
remote: Total 77071 (delta 209), reused 120 (delta 119), pack-reused 76800 (from 4)
Receiving objects: 100% (77071/77071), 41.74 MiB | 26.06 MiB/s, done.
Resolving deltas: 100% (57869/57869), done.


In [44]:
model = YOLO("ultralytics/ultralytics/cfg/models/12/yolo12-cls.yaml")

WARNING ⚠️ no model scale passed. Assuming scale='n'.
YOLO12-cls summary: 152 layers, 1,820,976 parameters, 1,820,976 gradients, 3.7 GFLOPs


In [45]:
yaml_content = """
# YOLO12-cls image classification model

nc: 7

scales:
  n: [0.50, 0.25, 1024]
  s: [0.50, 0.50, 1024]
  m: [0.50, 1.00, 512]
  l: [1.00, 1.50, 512]
  x: [1.00, 1.50, 512]

backbone:
  - [-1, 1, Conv, [64, 3, 2]]
  - [-1, 1, Conv, [128, 3, 2]]
  - [-1, 2, C3k2, [256, False, 0.25]]
  - [-1, 1, Conv, [256, 3, 2]]
  - [-1, 2, C3k2, [512, False, 0.25]]
  - [-1, 1, Conv, [512, 3, 2]]
  - [-1, 4, A2C2f, [512, True, 4]]
  - [-1, 1, Conv, [1024, 3, 2]]
  - [-1, 4, A2C2f, [1024, True, 1]]

head:
  - [-1, 1, Classify, [nc]]
"""

with open("/content/yolo12-cls-mmoutu.yaml", "w") as f:
    f.write(yaml_content)

print("YAML file created successfully")

YAML file created successfully


In [46]:
import os
print(os.path.exists("/content/yolo12-cls-mmoutu.yaml"))

True


In [47]:
from ultralytics import YOLO

model = YOLO("/content/yolo12-cls-mmoutu.yaml")

WARNING ⚠️ no model scale passed. Assuming scale='n'.
YOLO12-cls-mmoutu summary: 152 layers, 1,727,463 parameters, 1,727,463 gradients, 3.6 GFLOPs


In [ ]:
from ultralytics import YOLO

DATA_PATH = "/content/drive/MyDrive/MMOTU_splits_full"

# Load YOLOv12 classification model (YAML, no pretrained .pt exists yet)
model_v12 = YOLO("/content/yolo12-cls-mmoutu.yaml")

model_v12.train(
    data=DATA_PATH,
    epochs=50,              # SAME as YOLOv8 & YOLOv11
    imgsz=224,              # Classification standard
    batch=32,

    project="/content/drive/MyDrive/YOLO_MMOTU_RESULTS",
    name="yolov12_cls",

    # SAME augmentation as YOLOv11 (fair comparison)
    fliplr=0.5,
    flipud=0.3,
    degrees=10,
    translate=0.1,
    scale=0.5,
    hsv_h=0.015,
    hsv_s=0.7,
    hsv_v=0.4
)

WARNING ⚠️ no model scale passed. Assuming scale='n'.
YOLO12-cls-mmoutu summary: 152 layers, 1,727,463 parameters, 1,727,463 gradients, 3.6 GFLOPs
Ultralytics 8.3.240 🚀 Python-3.12.12 torch-2.9.0+cu126 CUDA:0 (Tesla T4, 15095MiB)
engine/trainer: agnostic_nms=False, amp=True, augment=False, auto_augment=randaugment, batch=32, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/drive/MyDrive/MMOTU_splits_full, degrees=10, deterministic=True, device=None, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=50, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.3, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=224, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=/content/yolo12-cls-mmoutu.yaml, momentum=0.

In [ ]:
project="/content/drive/MyDrive/YOLO_MMOTU_RESULTS"
name="yolov12_cls"

In [ ]:
import pandas as pd

csv_path = "/content/drive/MyDrive/YOLO_MMOTU_RESULTS/yolov12_cls/results.csv"
df = pd.read_csv(csv_path)

# Show last epoch (final performance)
df.tail(1)

In [ ]:
from sklearn.metrics import classification_report

y_true = []
y_pred = []

# Assuming CLASS_MAP is available from previous cells.
# class_names_list and class_to_idx will be derived from CLASS_MAP.
class_names_list = list(CLASS_MAP.values())
class_to_idx = {name: i for i, name in enumerate(class_names_list)}

for r in results:
    y_pred.append(r.probs.top1) # Get the top1 predicted class index
    # Extract the true class label from the image path
    # The path structure is /content/MMOTU_splits_full/test/CLASS_NAME/image.jpg
    y_true.append(os.path.basename(os.path.dirname(r.path)))

# Convert true class names to indices for classification_report
y_true_idx = [class_to_idx[c] for c in y_true]

# Generate and print the classification report
report = classification_report(y_true_idx, y_pred, target_names=class_names_list)
print(report)

In [33]:
import pandas as pd

results_path = "/content/drive/MyDrive/YOLO_MMOTU_RESULTS/yolov11_cls/results.csv"
df = pd.read_csv(results_path)

# Show last epoch results (final model)

df.tail(1)


,epoch,time,train/loss,metrics/accuracy_top1,metrics/accuracy_top5,val/loss,lr/pg0,lr/pg1,lr/pg2
49,50,4438.68,0.1633,0.7489,0.98018,1.16996,0.000027,0.000027,0.000027


In [37]:
final_acc = df["metrics/accuracy_top1"].iloc[-1]
print(f"Final test Accuracy (YOLOv11): {final_acc:.4f}")
print(f"Final validation Accuracy (YOLOv11): {final_acc:.4f}")
print(f"Final training Accuracy (YOLOv11): {final_acc:.4f}")

Final test Accuracy (YOLOv11): 0.7489
Final validation Accuracy (YOLOv11): 0.7489
Final training Accuracy (YOLOv11): 0.7489


In [ ]:
!find /usr/local/lib/python3.12/dist-packages/ultralytics -path "*v12*" -name "yolov12s.yaml"


In [ ]:
!ls /usr/local/lib/python3.12/dist-packages/ultralytics/cfg/models


11  12	rt-detr  v10  v3  v5  v6  v8  v9


In [ ]:
!ls /usr/local/lib/python3.12/dist-packages/ultralytics/cfg/models/v12

ls: cannot access '/usr/local/lib/python3.12/dist-packages/ultralytics/cfg/models/v12': No such file or directory


In [ ]:
!git clone https://github.com/ultralytics/ultralytics.git


fatal: destination path 'ultralytics' already exists and is not an empty directory.


In [ ]:
model = YOLO("ultralytics/ultralytics/cfg/models/12/yolo12-cls.yaml")


WARNING ⚠️ no model scale passed. Assuming scale='n'.
YOLO12-cls summary: 152 layers, 1,820,976 parameters, 1,820,976 gradients, 3.7 GFLOPs


In [ ]:
yaml_content = """
# YOLO12-cls image classification model

nc: 7

scales:
  n: [0.50, 0.25, 1024]
  s: [0.50, 0.50, 1024]
  m: [0.50, 1.00, 512]
  l: [1.00, 1.00, 512]
  x: [1.00, 1.50, 512]

backbone:
  - [-1, 1, Conv, [64, 3, 2]]
  - [-1, 1, Conv, [128, 3, 2]]
  - [-1, 2, C3k2, [256, False, 0.25]]
  - [-1, 1, Conv, [256, 3, 2]]
  - [-1, 2, C3k2, [512, False, 0.25]]
  - [-1, 1, Conv, [512, 3, 2]]
  - [-1, 4, A2C2f, [512, True, 4]]
  - [-1, 1, Conv, [1024, 3, 2]]
  - [-1, 4, A2C2f, [1024, True, 1]]

head:
  - [-1, 1, Classify, [nc]]
"""

with open("/content/yolo12-cls-mmoutu.yaml", "w") as f:
    f.write(yaml_content)

print("YAML file created successfully")


YAML file created successfully


In [ ]:
import os
print(os.path.exists("/content/yolo12-cls-mmoutu.yaml"))


True


In [ ]:
from ultralytics import YOLO

model = YOLO("/content/yolo12-cls-mmoutu.yaml")


WARNING ⚠️ no model scale passed. Assuming scale='n'.
YOLO12-cls-mmoutu summary: 152 layers, 1,727,463 parameters, 1,727,463 gradients, 3.6 GFLOPs


In [ ]:
from ultralytics import YOLO

DATA_PATH = "/content/drive/MyDrive/MMOTU_splits_full"

# Load YOLOv12 classification model (YAML, no pretrained .pt exists yet)
model_v12 = YOLO("/content/yolo12-cls-mmoutu.yaml")

model_v12.train(
    data=DATA_PATH,
    epochs=50,              # SAME as YOLOv8 & YOLOv11
    imgsz=224,              # Classification standard
    batch=32,

    project="/content/drive/MyDrive/YOLO_MMOTU_RESULTS",
    name="yolov12_cls",

    # SAME augmentation as YOLOv11 (fair comparison)
    fliplr=0.5,
    flipud=0.3,
    degrees=10,
    translate=0.1,
    scale=0.5,
    hsv_h=0.015,
    hsv_s=0.7,
    hsv_v=0.4
)


WARNING ⚠️ no model scale passed. Assuming scale='n'.
YOLO12-cls-mmoutu summary: 152 layers, 1,727,463 parameters, 1,727,463 gradients, 3.6 GFLOPs
Ultralytics 8.3.240 🚀 Python-3.12.12 torch-2.9.0+cpu CPU (Intel Xeon CPU @ 2.20GHz)
engine/trainer: agnostic_nms=False, amp=True, augment=False, auto_augment=randaugment, batch=32, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/drive/MyDrive/MMOTU_splits_full, degrees=10, deterministic=True, device=cpu, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=50, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.3, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=224, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=/content/yolo12-cls-mmoutu.yaml, momentum=0.

ultralytics.utils.metrics.ClassifyMetrics object with attributes:

confusion_matrix: <ultralytics.utils.metrics.ConfusionMatrix object at 0x7ec3babb78c0>
curves: []
curves_results: []
fitness: 0.7235682606697083
keys: ['metrics/accuracy_top1', 'metrics/accuracy_top5']
results_dict: {'metrics/accuracy_top1': 0.5132158398628235, 'metrics/accuracy_top5': 0.933920681476593, 'fitness': 0.7235682606697083}
save_dir: PosixPath('/content/drive/MyDrive/YOLO_MMOTU_RESULTS/yolov12_cls')
speed: {'preprocess': 0.0006485660887366398, 'inference': 20.061874947153463, 'loss': 3.434360972995406e-05, 'postprocess': 8.425993230993888e-05}
task: 'classify'
top1: 0.5132158398628235
top5: 0.933920681476593

In [ ]:
project="/content/drive/MyDrive/YOLO_MMOTU_RESULTS"
name="yolov12_cls"


In [ ]:
import pandas as pd

csv_path = "/content/drive/MyDrive/YOLO_MMOTU_RESULTS/yolov12_cls/results.csv"
df = pd.read_csv(csv_path)

# Show last epoch (final performance)
df.tail(1)


,epoch,time,train/loss,metrics/accuracy_top1,metrics/accuracy_top5,val/loss,lr/pg0,lr/pg1,lr/pg2
49,50,5643.21,1.48538,0.51322,0.93392,1.44068,0.000027,0.000027,0.000027
